회귀 손실함수 비교

In [ ]:
# 출력층에서는 별도의 활성화 함수(ReLU 등)를 씌우지 않아야만 하니?
# 1 단계 완료
# 네, 목적에 따라 다릅니다. 출력층에서 활성화 함수를 씌울지 말지는 풀고자 하는 문제의 종류에 따라 결정됩니다.
# 항상 활성화 함수를 씌우지 않아야 하는 것은 아니며, 다음과 같은 기준을 따릅니다.
# 1. 활성화 함수를 씌우지 않는 경우 (항등 함수)
# 회귀(Regression) 문제: 온도, 가격, 나이 등 연속적인 '수치' 자체를 예측해야 할 때는 출력층에 활성화 함수를 
# 적용하지 않습니다. 선형 계층의 결괏값(음수부터 양수까지 무한대의 범위)을 그대로 출력해야 하기 때문입니다. 
# 작성하신 코드도 이 경우에 해당합니다.
# 2. 활성화 함수를 씌우는 경우
# 이진 분류(Binary Classification): 스팸 메일 여부처럼 두 가지 중 하나를 예측할 때는 출력값이 0~1 
# 사이의 확률이어야 하므로 출력층에 시그모이드(Sigmoid) 함수를 씌웁니다.
# 다중 분류(Multi-class Classification): 개, 고양이, 새 등 세 개 이상의 클래스 중 하나를 고를 때는 각 
# 클래스에 대한 확률의 합이 1이 되도록 출력층에 소프트맥스(Softmax) 함수를 씌웁니다.
# 단, PyTorch에서는 손실 함수 내부(예: nn.BCEWithLogitsLoss나 nn.CrossEntropyLoss)에 이미 
# 시그모이드나 소프트맥스 연산이 포함되어 있는 경우가 많아서, 코드 상으로는 출력층에 활성화 함수를 적지 않고 
# 그대로 넘기는 경우도 흔합니다.



# 정규화
# StandardScaler는 머신러닝에서 데이터의 스케일(단위나 범위)을 맞추기 위해 사용하는 대표적인 데이터 스케일링(Data Scaling) 
# 기법 중 하나입니다.
# 1. StandardScaler란 무엇인가요?
# 데이터의 피처(Feature)들을 평균이 0이고, 표준편차가 1인 정규분포 형태로 변환해주는 도구입니다.
# 2. 왜 필요한가요?
# 머신러닝 모델은 데이터의 '크기'에 민감하게 반응합니다.
# 예를 들어, 어떤 변수는 0~1 사이의 값을 가지고(예: 비율), 어떤 변수는 0~100,000의 값을 가질 때(예: 가격), 
# 모델은 숫자가 큰 '가격' 변수가 더 중요하다고 착각할 수 있습니다.
# 따라서 모든 변수를 평등하게 평가하고, 모델이 특정 변수에 과도하게 영향을 받는 것을 막기 위해 모든 데이터의 
# 범위를 비슷한 수준(평균 0, 분산 1)으로 통일시켜 주는 과정이 필요합니다. 이 과정을 거치면 경사 하강법 등을 사용하는 
# 모델의 학습 속도와 안정성도 크게 향상됩니다.
# 3. 언제 쓰나요?
# 거리 기반 알고리즘을 쓸 때: KNN, SVM(서포트 벡터 머신) 등 거리를 계산하여 예측하는 알고리즘에서는 스케일링이 필수적입니다.
# 선형 회귀, 로지스틱 회귀를 쓸 때: 규제(Regularization)를 공평하게 적용하기 위해 사용합니다.
# 딥러닝(신경망) 모델을 학습할 때: 입력 데이터의 분포를 균일하게 맞춰 학습을 안정적으로 만들기 위해 사용합니다.
# 데이터가 정규분포를 따를 때: 데이터가 정규분포에 가까울 때 가장 효과적입니다. (단, 이상치가 너무 많다면 중앙값과 
# IQR을 사용하는 RobustScaler가 더 적합할 수 있습니다 ).

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
#재현성 위한 시드 고정
torch.manual_seed(42)
np.random.seed(42)

In [4]:
# 간단한 예측값과 실제값
y_true = torch.tensor([10.0, 20.0, 30.0])
y_pred = torch.tensor([12.0, 19.0, 35.0])

In [6]:
# mse 계산
mse_loss = nn.MSELoss()
mse_loss(y_pred, y_true)
mse_value = mse_loss(y_pred, y_true)
print(mse_value)

tensor(10.)


In [16]:
# mse 계산
mse_loss = nn.MSELoss()
mse_loss(y_pred, y_true)
mse_value = mse_loss(y_pred, y_true)
print(mse_value)
print(np.sqrt(mse_value)) # rmse

tensor(10.)
tensor(3.1623)


/tmp/ipykernel_175/2638374304.py:6: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  print(np.sqrt(mse_value))


In [14]:
# mae 계산
mae_loss = nn.L1Loss()
mae_loss(y_pred, y_true)
mae_value = mae_loss(y_pred, y_true)
print(mae_value)

tensor(2.6667)


In [8]:
# huber loss 계산
huber_loss = nn.HuberLoss(delta=1.0)
huber_value = huber_loss(y_pred, y_true)
print(huber_value)

tensor(2.1667)


In [12]:
print(y_pred.numpy()) # 예측값
print(y_true.numpy()) # 실제값
print((y_pred-y_true).numpy()) # 오차(error)

[12. 19. 35.]
[10. 20. 30.]
[ 2. -1.  5.]


In [15]:
print(mse_value.numpy())
print(mae_value.numpy())
print(huber_value.numpy())

10.0
2.6666667
2.1666667


In [17]:
# 수동 계산으로 검증
(y_pred - y_true).numpy() # loss

array([ 2., -1.,  5.], dtype=float32)

In [20]:
errors = (y_pred - y_true).numpy()

print(np.mean(errors ** 2)) # mse
print(np.mean(np.abs(errors))) # mae

10.0
2.6666667


이상치가 있을 때 손실함수 비교

In [21]:
# 정상 데이터 + 이상치
y_true_with_outlier = torch.tensor([10.0, 20.0, 30.0, 40.0, 50.0])
y_pred_normal = torch.tensor([11.0, 19.0, 31.0, 39.0, 51.0])  # 정상 예측
y_pred_with_outlier = torch.tensor([11.0, 19.0, 31.0, 100.0, 51.0])  # 이상치 포함

In [23]:
# 정상 예측
print("실제: ", y_true_with_outlier.numpy())
print("예측: ", y_pred_normal.numpy())

실제:  [10. 20. 30. 40. 50.]
예측:  [11. 19. 31. 39. 51.]


In [24]:
# 각 회귀손실함수 적용
mse_normal = mse_loss(y_pred_normal, y_true_with_outlier)
mae_normal = mae_loss(y_pred_normal, y_true_with_outlier)
huber_normal = huber_loss(y_pred_normal, y_true_with_outlier)

print(mse_normal)
print(mae_normal)
print(huber_normal)

tensor(1.)
tensor(1.)
tensor(0.5000)


In [25]:
# 이상치 포함 예측
print("실제: ", y_true_with_outlier.numpy())
print("예측: ", y_pred_with_outlier.numpy())
print("오차: ", (y_true_with_outlier - y_pred_with_outlier).numpy())

실제:  [10. 20. 30. 40. 50.]
예측:  [ 11.  19.  31. 100.  51.]
오차:  [ -1.   1.  -1. -60.  -1.]


In [26]:
mse_outlier = mse_loss(y_pred_with_outlier, y_true_with_outlier)
mae_outlier = mae_loss(y_pred_with_outlier, y_true_with_outlier)
huber_outlier = huber_loss(y_pred_with_outlier, y_true_with_outlier)

In [27]:
print(mse_outlier)
print(mae_outlier)
print(huber_outlier)

tensor(720.8000)
tensor(12.8000)
tensor(12.3000)


In [31]:
# 분석 시 이렇게 활용
# 증가율 비교
print(f'증가율 : {mse_outlier / mse_normal:.2f}배 증가')
print(f'증가율 : {mae_outlier / mae_normal:.2f}배 증가')
print(f'증가율 : {huber_outlier / huber_normal:.2f}배 증가')

증가율 : 720.80배 증가
증가율 : 12.80배 증가
증가율 : 24.60배 증가


In [32]:
# 분석 결과
# mse는 이상치에 매우 민감한 것을 발견함. (제곱 때문)
# mae는 이상치에 강건(robust) (절대값, 즉 크기만 고려)
# huber 중간 (작은오차는 제곱, 큰 오차는 선형)

손실함수별 학습 비교

In [34]:
# 데이터 생성 (일부러 이상치 추가)
X, y = make_regression(n_samples=500, n_features=10, noise=10.0, random_state=42)

In [35]:
# 이상치 추가 (10% 데이터에 큰 노이즈)
n_outliers = int(0.1 * len(y))
outlier_indices = np.random.choice(len(y), n_outliers, replace=False)
y[outlier_indices] += np.random.randn(n_outliers) * 50  # 큰 노이즈

In [36]:
# 데이터 분할
X_train, X_test, y_train, y_test =\
train_test_split(X, y, test_size = 0.2, random_state=42)

In [ ]:
# 정규화
# StandardScaler는 머신러닝에서 데이터의 스케일(단위나 범위)을 맞추기 위해 사용하는 대표적인 데이터 스케일링(Data Scaling) 
# 기법 중 하나입니다.
# 1. StandardScaler란 무엇인가요?
# 데이터의 피처(Feature)들을 평균이 0이고, 표준편차가 1인 정규분포 형태로 변환해주는 도구입니다.
# 2. 왜 필요한가요?
# 머신러닝 모델은 데이터의 '크기'에 민감하게 반응합니다.
# 예를 들어, 어떤 변수는 0~1 사이의 값을 가지고(예: 비율), 어떤 변수는 0~100,000의 값을 가질 때(예: 가격), 
# 모델은 숫자가 큰 '가격' 변수가 더 중요하다고 착각할 수 있습니다.
# 따라서 모든 변수를 평등하게 평가하고, 모델이 특정 변수에 과도하게 영향을 받는 것을 막기 위해 모든 데이터의 
# 범위를 비슷한 수준(평균 0, 분산 1)으로 통일시켜 주는 과정이 필요합니다. 이 과정을 거치면 경사 하강법 등을 사용하는 
# 모델의 학습 속도와 안정성도 크게 향상됩니다.
# 3. 언제 쓰나요?
# 거리 기반 알고리즘을 쓸 때: KNN, SVM(서포트 벡터 머신) 등 거리를 계산하여 예측하는 알고리즘에서는 스케일링이 필수적입니다.
# 선형 회귀, 로지스틱 회귀를 쓸 때: 규제(Regularization)를 공평하게 적용하기 위해 사용합니다.
# 딥러닝(신경망) 모델을 학습할 때: 입력 데이터의 분포를 균일하게 맞춰 학습을 안정적으로 만들기 위해 사용합니다.
# 데이터가 정규분포를 따를 때: 데이터가 정규분포에 가까울 때 가장 효과적입니다. (단, 이상치가 너무 많다면 중앙값과 
# IQR을 사용하는 RobustScaler가 더 적합할 수 있습니다 ).
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
# print(X_train)
X_test = scaler.transform(X_test)

In [47]:
# 텐서변환
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)

X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)


# print(X_train_t.shape)
# print(y_train_t.shape)
# print(y_train_t)

In [49]:
# 훈련 데이터
print(X_train.shape)

# 이상치
print(n_outliers)

(400, 10)
50


In [50]:
# 간단한 회귀 모델
class RegressionModel(nn.Module):
    """간단한 회귀 신경망"""
    def __init__(self):
        super(RegressionModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [51]:
# 세 가지 손실함수로 각각 학습
loss_functions = {
    'MSE': nn.MSELoss(),
    'MAE': nn.L1Loss(),
    'Huber': nn.HuberLoss(delta=1.0)
}

In [52]:
results = {}

for loss_name, criterion in loss_functions.items():
    print(f"\n{loss_name}로 학습 중...")

    # 모델 초기화
    model = RegressionModel()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # 학습
    train_losses = []
    num_epochs = 100

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()

        output = model(X_train_t)
        loss = criterion(output, y_train_t)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # 테스트
    model.eval()
    with torch.no_grad():
        test_pred = model(X_test_t)

        # 모든 지표로 평가
        test_mse = nn.MSELoss()(test_pred, y_test_t).item()
        test_mae = nn.L1Loss()(test_pred, y_test_t).item()
        test_huber = nn.HuberLoss()(test_pred, y_test_t).item()

    results[loss_name] = {
        'train_losses': train_losses,
        'test_mse': test_mse,
        'test_mae': test_mae,
        'test_huber': test_huber,
        'predictions': test_pred
    }

    print(f"  최종 훈련 손실: {train_losses[-1]:.4f}")
    print(f"  테스트 MSE: {test_mse:.4f}")
    print(f"  테스트 MAE: {test_mae:.4f}")


MSE로 학습 중...
  최종 훈련 손실: 140.4987
  테스트 MSE: 1344.0741
  테스트 MAE: 25.7575

MAE로 학습 중...
  최종 훈련 손실: 6.9503
  테스트 MSE: 1276.3252
  테스트 MAE: 25.1702

Huber로 학습 중...
  최종 훈련 손실: 6.4076
  테스트 MSE: 1340.1527
  테스트 MAE: 26.2605
